# Primate Vocalization Detection — Local Run

Run the whole pipeline on your own machine with **minimal path configuration**.

## Setup (one time)

1. **Install the environment** by following [`SETUP.md`](../SETUP.md) in the repo
   root — it creates a conda environment with the exact tested versions
   (Python 3.12, TensorFlow 2.20, Keras 3.13.2). Do **not** `conda install
   tensorflow`; that installs an old version that cannot load the model.
2. **Download the pretrained model** — see the "Pretrained Model" section in
   [`README.md`](../README.md). Place `best_model_v12.h5` at
   `data/outputs/models/best_model_v12.h5`. (Or skip this to train from scratch.)
3. Drop your audio into the ready-made folders under [`data/`](../data/README.md):
   - `data/species/<call-type>/` — labelled call clips (training positives)
   - `data/background/<...>/` — negative clips
   - `data/long_audio/` — continuous recordings to run detection on
     *(or leave them on an external drive — see `FIELD_RECORDINGS_DIR` in Step 1)*
   - See `data/README.md` for exactly which folder each file goes in.
4. Run the cells below **in order**.

The first cell points `PRIMATE_DATA_ROOT` at this repo's `data/` folder for you. To run detection on field recordings that live on an external drive **without copying them**, set `FIELD_RECORDINGS_DIR` in Step 1 to that folder — everything else (models, training data, outputs) stays under local `data/`. The model uses the production **V12** head (`temporal_freqpos`).

## Running across several machines (data on different drives)

Each computer processes the recordings on **its own drive**:

1. Copy `best_model_v12.h5` into this machine's `data/outputs/models/` (small file).
2. In **Step 1** below, uncomment the `FIELD_RECORDINGS_DIR` line for **this** machine's drive (comment out the rest).
3. Run the cells. Detections are written to this machine's `data/outputs/detections/`.
4. After every machine finishes, gather the per-machine `detections/` outputs and combine them.

## Step 1 — Set up paths and load config

Locates the repo root automatically (works whether you launch Jupyter from the repo root or from this `main_pipeline_notebooks/` folder), points the pipeline at `data/`, and selects the V12 head.

In [ ]:
import os, sys, importlib
from pathlib import Path

def find_repo_root(start=None):
    """Walk up from the working directory until we find src/config.py + data/."""
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'src' / 'config.py').is_file() and (cand / 'data').is_dir():
            return cand
    raise RuntimeError('Run this notebook from inside the cloned repo '
                       '(it needs src/config.py and the data/ folder).')

REPO = find_repo_root()
os.environ['PRIMATE_DATA_ROOT']     = str(REPO / 'data')
os.environ['PRIMATE_MODEL_POOLING'] = 'temporal_freqpos'   # production V12 head

# --- Field recordings location -------------------------------------------------
# By default the pipeline looks for continuous recordings in data/long_audio/.
# If your field recordings live on an external drive and you DON'T want to copy
# them, point FIELD_RECORDINGS_DIR at that folder. The scan is recursive, so a
# parent folder containing many station subfolders (CL C1 SBL, CBG1 FSCA1, ...)
# is processed in one pass. Set to None to use data/long_audio/ instead.
# Only the field-recording path is redirected — models, training data and all
# outputs still live in the local data/ folder.
# >>> THIS MACHINE: keep ONE line uncommented, comment out the others <<<
FIELD_RECORDINGS_DIR = r'E:\SoundForestLab\Gabon CNN BirdNET\acoustic_data'   # machine 1 / E: drive
# FIELD_RECORDINGS_DIR = r'D:\Gabon\acoustic_data'                             # machine 2 / D: drive   (edit to the real path)
# FIELD_RECORDINGS_DIR = r'F:\acoustic_data'                                    # machine 3 / F: drive   (edit to the real path)
# FIELD_RECORDINGS_DIR = r'/Volumes/<DRIVE>/acoustic_data'                       # macOS external drive   (edit)
# FIELD_RECORDINGS_DIR = None                                                    # fall back to data/long_audio/
if FIELD_RECORDINGS_DIR:
    os.environ['PRIMATE_LONG_AUDIO_ROOT'] = FIELD_RECORDINGS_DIR
# -------------------------------------------------------------------------------

SRC = str(REPO / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import config
importlib.reload(config)   # re-read env vars in case a stale copy was cached
print('Repo root      :', REPO)
print('Data root      :', config.DRIVE_ROOT)
print('Long audio root:', config.LONG_AUDIO_ROOT)
config.print_config_summary()

## Step 1.5 — GPU check (automatic)

TensorFlow will use a GPU if one is available. No code changes needed — this cell just detects and reports what hardware you have. If you see "No GPU", everything still works (just slower for training). See `SETUP.md § Optional: GPU acceleration` if you want to enable it.

In [ ]:
import platform, tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s) found: {len(gpus)}')
    for g in gpus:
        print(f'  - {g.name}')
    print('TensorFlow will use GPU automatically.')
else:
    system = platform.system()
    chip = platform.processor() or platform.machine()
    print(f'No GPU detected (running on CPU — {system} {chip}).')
    print('Inference works fine on CPU; training is slower.')
    if system == 'Darwin' and 'arm' in platform.machine().lower():
        print('\nTip: you have Apple Silicon — run "pip install tensorflow-metal"')
        print('     to enable GPU acceleration, then restart the kernel.')
    elif system == 'Windows':
        print('\nNote: TensorFlow on Windows is CPU-only since v2.11.')
        print('      For GPU on Windows, use WSL2 (see SETUP.md).')

print(f'\nTensorFlow {tf.__version__} | Python {platform.python_version()} | {platform.system()} {platform.machine()}')

## Step 2 — Data sanity check

Loads every WAV under `data/species/` and `data/background/` and prints how many clips per class. **If a class shows 0 clips**, the folder is empty or the name doesn't match — see `data/README.md`. (Empty optional folders just print a warning and are skipped.)

In [ ]:
import data_loader
importlib.reload(data_loader)

species_data = data_loader.load_species_data()
background_data = data_loader.load_background_data()
data_loader.print_data_summary(species_data, background_data)

total_species = sum(len(v) for v in species_data.values())
assert total_species > 0, ('No species clips found. Drop WAV files into '
                           'data/species/<...>/  (see data/README.md).')
assert len(background_data) > 0, ('No background clips found. Drop WAV files '
                                  'into data/background/<...>/.')
print(f'\nOK: {total_species} species clips, {len(background_data)} background clips.')

## Step 3 — Train, or load an existing model

By default this **loads** `data/outputs/models/best_model_v12.h5` (falling back to `best_model.h5` if the V12 weights aren't there yet); if neither exists it **trains** from scratch (two-stage: frozen base → fine-tune; up to ~20–60 min on CPU/GPU). Set `FORCE_RETRAIN = True` to always retrain.

Because the V12 head contains the custom `FrequencyCoord` layer, the model is loaded with `model.load_trained_model(...)`, **not** raw `tf.keras.models.load_model(...)`.

In [ ]:
import model as model_lib
importlib.reload(model_lib)

FORCE_RETRAIN = False   # set True to retrain from scratch

# Prefer the production V12 model; fall back to best_model.h5 if you haven't
# copied the V12 weights in yet. Drop your .h5 into data/outputs/models/.
candidates = ['best_model_v12.h5', 'best_model.h5']
best_model_path = next(
    (os.path.join(config.MODEL_SAVE_DIR, n)
     for n in candidates if os.path.exists(os.path.join(config.MODEL_SAVE_DIR, n))),
    os.path.join(config.MODEL_SAVE_DIR, candidates[0]))

if FORCE_RETRAIN or not os.path.exists(best_model_path):
    import train
    importlib.reload(train)
    print('Training model (temporal_freqpos head)...')
    model = train.run_complete_training_pipeline()
    print('\nTraining done. Best model saved to:', best_model_path)
else:
    print('Loading existing model:', best_model_path)
    model = model_lib.load_trained_model(best_model_path)   # passes custom_objects for FrequencyCoord
    model_lib.print_model_summary(model)

## Step 4 — Detect in the long recordings

Slides a 2 s window over every recording in **`config.LONG_AUDIO_ROOT`**, classifies each window, applies non-maximum suppression, and writes one CSV per file to `data/outputs/detections/`.

By default that root is `data/long_audio/`. If you set `FIELD_RECORDINGS_DIR` in Step 1, it points straight at your external drive instead (e.g. `acoustic_data/`) and the scan recurses into every station subfolder — **no copying required**. `TIME_FILTER = False` (the default) processes **all** recordings. Set it to `True` only if you want to restrict to the dawn survey window (05:30–10:30).

In [ ]:
import detection
importlib.reload(detection)

# Process every recording by default. Set TIME_FILTER = True only if you want to
# restrict to the dawn survey window (config.TIME_FILTER_START–TIME_FILTER_END,
# default 05:30–10:30) and skip the rest of the day.
TIME_FILTER = False

print('Reading recordings from:', config.LONG_AUDIO_ROOT)
all_detections = detection.process_all_long_audio_files(model, time_filter=TIME_FILTER)

total = sum(len(df) for df in all_detections.values())
print(f'\n{total} detections across {len(all_detections)} files.')
print('CSVs written to:', config.DETECTION_OUTPUT_DIR)
if total == 0:
    print('\n(No detections. Check FIELD_RECORDINGS_DIR in Step 1, '
          'or lower config.DETECTION_CONFIDENCE_THRESHOLD.)')

## Step 5 — Export detection clips for manual review

Cuts a short WAV around each detection (0.5 s padding) into `data/outputs/detected_clips/<species>/<station>/` so you can listen through them.

> **Ran detection on another machine?** If you copied the per-station `*_detections.csv` folders into `data/outputs/detections/` instead of running Step 4 here, this cell rebuilds the detection list straight from those CSVs — just make sure the matching source recordings are reachable under your `FIELD_RECORDINGS_DIR` (Step 1) so the clips can be cut.

In [ ]:
import utils
importlib.reload(utils)

# Step 4 fills `all_detections` in memory. If you skipped Step 4 this session
# (e.g. detection ran on another machine and you copied the *_detections.csv
# folders into data/outputs/detections/), rebuild it from those CSVs instead.
if 'all_detections' not in globals():
    all_detections = utils.load_all_detections_from_disk()

total = sum(len(df) for df in all_detections.values())

if total > 0:
    clips_dir = utils.extract_all_detected_clips(
        all_detections, padding=0.5, organize_by_species=True)
    print('Review clips at:', clips_dir)
else:
    print('Nothing to export — no detections yet.')

## Done

Outputs are under `data/outputs/`:
- `models/best_model_v12.h5` — the trained V12 model
- `detections/*.csv` — per-file detections
- `detected_clips/<species>/*.wav` — clips to review

To iterate: review the clips, move confirmed false positives into a `data/background/` folder, and re-run Step 3 with `FORCE_RETRAIN = True`.